# Chapter 03: Attention dimensions and projection layouts


Attention is mostly disciplined shape manipulation. These exercises
focus on matrix-storage conventions, concatenating independent heads,
and validating the divisibility condition that makes a multi-head split
possible.


    This notebook is an independently written practice companion. It
    does not reproduce the book's prose, figures, data, or solutions.
    Read the [Chinese chapter](https://github.com/skindhu/Build-A-Large-Language-Model-CN/blob/main/cn-Book/3.%E5%AE%9E%E7%8E%B0%E6%B3%A8%E6%84%8F%E5%8A%9B%E6%9C%BA%E5%88%B6.md) for the complete narrative
    and exercise statements, and use the
    [canonical MIT companion code](https://github.com/rasbt/LLMs-from-scratch)
    as an additional reference.

    **Prerequisites:** Python 3.10+ and PyTorch 2.2+. Every example is
    synthetic, deterministic, CPU-friendly, and makes no network call.

    **Workflow**

    1. Read the concept and trace the shapes by hand.
    2. Replace one `LLM_TODO` and its `NotImplementedError`.
    3. Run that exercise's check cell.
    4. Explain the result before moving to the next exercise.
    5. Compare with `solution.py` only after making a serious attempt.

In [ ]:
from __future__ import annotations

import math

import torch
from torch import Tensor, nn

## Exercise implementation A: Simple self-attention

    **Objective.** Implement the complete score-normalize-mix path using the input as Q, K, and V.


Self-attention creates a context-dependent representation for every
token. In the simplest version, the same input tensor supplies queries,
keys, and values. Pairwise scores are
$S=XX^T/\sqrt{D}$, row-wise weights are
$A=\operatorname{softmax}(S)$, and context is $C=AX$.

For input shape `[B,T,D]`, scores and weights have shape `[B,T,T]`.
Each row of `A` sums to one and describes how one query position mixes
all value positions. The scale $\sqrt{D}$ keeps score magnitude from
growing unchecked as channel width increases.


    ### Data-flow walkthrough


Start with `[B,T,D]`. Transpose only the final two axes to obtain
`[B,D,T]`; batch matrix multiplication then yields `[B,T,T]`. Applying
weights `[B,T,T]` to values `[B,T,D]` returns `[B,T,D]`.


    ### Hints and common mistakes


Softmax belongs on the key axis (`dim=-1`). Do not transpose the batch
axis. Verify both output shape and row sums; a shape-only test cannot
detect normalization on the wrong axis.

In [ ]:
def simple_self_attention(inputs: Tensor) -> tuple[Tensor, Tensor]:
    # ATTENTION_TODO_CH03_00_SIMPLE
    raise NotImplementedError(
        "compute scaled scores, weights, and context"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
torch.manual_seed(1)
implementation_inputs = torch.randn(2, 5, 4)
context, weights = simple_self_attention(implementation_inputs)
assert context.shape == implementation_inputs.shape
assert weights.shape == (2, 5, 5)
assert torch.allclose(
    weights.sum(dim=-1),
    torch.ones_like(weights.sum(dim=-1)),
)
print("Simple attention passed")

## Exercise implementation B: Learned causal Q/K/V attention

    **Objective.** Add learned projections and prevent every position from reading future tokens.


Learned attention projects the input into queries, keys, and values:
$Q=XW_Q$, $K=XW_K$, and $V=XW_V$. Scores are
$QK^T/\sqrt{D_h}$. A causal decoder must hide score entries where key
position $j$ is greater than query position $i$.

Store the upper-triangular mask as a registered Boolean buffer. It then
follows the module across devices without becoming a trainable
parameter. Replace masked scores with negative infinity before softmax,
making their resulting probability zero.


    ### Data-flow walkthrough


Q, K, and V each have `[B,T,Dout]`; scores have `[B,T,T]`. Slice a
`[context,context]` mask to `[:T,:T]`. The output after weights times V
is `[B,T,Dout]`. Changing tokens at positions 3 and 4 must not alter
outputs at positions 0 through 2.


    ### Hints and common mistakes


Mask before softmax, scale by key width, and reject sequences longer
than the configured context. Dropout applies to attention weights during
training; use dropout zero for deterministic checks.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        context_length: int,
        dropout: float = 0.0,
        qkv_bias: bool = False,
    ) -> None:
        super().__init__()
        # ATTENTION_TODO_CH03_00_CAUSAL
        raise NotImplementedError(
            "create projections, dropout, and a causal-mask buffer"
        )

    def forward(self, inputs: Tensor) -> Tensor:
        raise NotImplementedError(
            "project, mask, normalize, and mix values"
        )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
causal = CausalSelfAttention(4, 4, context_length=5)
changed = implementation_inputs.clone()
changed[:, 3:] += 100
original_prefix = causal(implementation_inputs)[:, :3]
changed_prefix = causal(changed)[:, :3]
assert original_prefix.shape == (2, 3, 4)
assert torch.allclose(original_prefix, changed_prefix, atol=1e-5, rtol=1e-5)
print("Causal attention passed")

## Exercise implementation C: Efficient multi-head causal attention

    **Objective.** Project all heads together, split a head axis, attend in parallel, and merge.


Multi-head attention uses several smaller attention spaces in parallel.
One dense projection still produces width $D_{out}$, then reshaping
exposes `H` heads of width $D_h=D_{out}/H`. The score tensor becomes
`[B,H,T,T]`, so every head has its own attention matrix.

After attention, transpose `[B,H,T,Dh]` to `[B,T,H,Dh]` and merge the
final two axes back to `[B,T,Dout]`. An output projection can then mix
information across heads. The causal mask broadcasts over batch and
head dimensions.


    ### Data-flow walkthrough


With `Dout=8` and `H=2`, each head width is four. Q/K/V move from
`[B,T,8]` to `[B,2,T,4]`. Scores are `[B,2,T,T]`; merged context returns
to `[B,T,8]`.


    ### Hints and common mistakes


Require exact divisibility. Call `.contiguous()` after transposing and
before `view`, because transpose changes strides. Scale by per-head
width, not the full output width.

In [ ]:
class MultiHeadCausalAttention(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        context_length: int,
        num_heads: int,
        dropout: float = 0.0,
        qkv_bias: bool = False,
    ) -> None:
        super().__init__()
        # ATTENTION_TODO_CH03_00_MULTIHEAD
        raise NotImplementedError(
            "create projections and record head dimensions"
        )

    def forward(self, inputs: Tensor) -> Tensor:
        raise NotImplementedError(
            "split heads, attend causally, and merge heads"
        )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
multihead = MultiHeadCausalAttention(
    4, 8, context_length=5, num_heads=2
)
output = multihead(implementation_inputs)
changed_output = multihead(changed)
assert output.shape == (2, 5, 8)
assert torch.allclose(
    output[:, :3], changed_output[:, :3], atol=1e-5, rtol=1e-5
)
print("Multi-head causal attention passed")

## Exercise 3.1: Transfer projection weights

    **Objective.** Copy a raw matrix projection into `nn.Linear` without changing outputs.


Suppose a hand-written layer stores $W_{raw}$ with shape
`[in_features, out_features]` and computes $XW_{raw}$. PyTorch's
`nn.Linear` stores its weight as `[out_features, in_features]` and
computes $XW_{linear}^{T}+b$. Equivalent behavior therefore requires
$W_{linear}=W_{raw}^{T}$.

This is a storage-layout issue, not a mathematical change to the
projection. Copy under `torch.no_grad()` so setup is not added to the
autograd graph. A bias must be zeroed when the raw projection has none.


    ### Data-flow walkthrough


With `X: [B,T,Din]` and raw weight `[Din,Dout]`, the raw output is
`[B,T,Dout]`. `nn.Linear` accepts the same input but internally
transposes its `[Dout,Din]` parameter. Matching numerical output is a
stronger check than comparing shapes.


    ### Hints and common mistakes


Do not assign a new tensor to `linear.weight`; copy into the existing
parameter. Check both dimensions before transposing. Forgetting the bias
can produce a plausible but shifted result.

In [ ]:
class RawProjection(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(
            torch.empty(in_features, out_features)
        )

    def forward(self, inputs: Tensor) -> Tensor:
        return inputs @ self.weight


def transfer_raw_to_linear(
    raw_weight: Tensor, linear: nn.Linear
) -> None:
    # LLM_TODO_CH03_01_TRANSFER
    raise NotImplementedError("copy using nn.Linear's storage layout")

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
torch.manual_seed(7)
raw = RawProjection(4, 3)
nn.init.normal_(raw.weight)
linear = nn.Linear(4, 3)
transfer_raw_to_linear(raw.weight, linear)
sample = torch.randn(2, 5, 4)
assert torch.allclose(raw(sample), linear(sample))
print("Exercise 3.1 passed")

## Exercise 3.2: Concatenate multiple heads

    **Objective.** Combine head outputs on the feature axis and reason about total width.


A simple teaching implementation can run several attention heads
independently and concatenate their outputs. If each head produces
`[B,T,Dh]` and there are $H$ heads, concatenation on the last axis gives
`[B,T,H\cdot Dh]`.

Concatenating on the sequence axis would instead alter token count,
destroying the alignment between positions. The batch and sequence
dimensions must remain unchanged because every head describes the same
examples and token positions.


    ### Data-flow walkthrough


Two toy heads mapping width 4 to width 2 each receive `[2,5,4]`. Each
returns `[2,5,2]`; concatenation along `dim=-1` returns `[2,5,4]`.
To preserve a desired model width $D$, choose $Dh=D/H$.


    ### Hints and common mistakes


Require at least one head and register heads in `nn.ModuleList`, or
parameters will not appear in the wrapper's state dictionary. Use
`dim=-1`, not `dim=1`.

In [ ]:
class MultiHeadWrapper(nn.Module):
    def __init__(self, heads: list[nn.Module]) -> None:
        super().__init__()
        if not heads:
            raise ValueError("at least one head is required")
        self.heads = nn.ModuleList(heads)

    def forward(self, inputs: Tensor) -> Tensor:
        # LLM_TODO_CH03_02_MULTIHEAD
        raise NotImplementedError(
            "concatenate head outputs on the feature axis"
        )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
wrapper = MultiHeadWrapper([nn.Linear(4, 2), nn.Linear(4, 2)])
output = wrapper(torch.randn(2, 5, 4))
assert output.shape == (2, 5, 4)
print("Exercise 3.2 passed")

## Exercise 3.3: Validate per-head dimensions

    **Objective.** Check that an embedding width can be split evenly across attention heads.


Efficient multi-head attention reshapes a model-width axis $D$ into
`[H, Dh]`, where $Dh=D/H$. A reshape cannot create or discard elements,
so $D$ must be divisible by $H$. GPT-2 small uses $D=768$ and $H=12$,
yielding $Dh=64$.

The full flow for a projected tensor is
`[B,T,D] -> [B,T,H,Dh] -> [B,H,T,Dh]`. The transpose places heads before
tokens so batched attention scores become `[B,H,T,T]`.


    ### Data-flow walkthrough


`768 / 12 = 64`, so each head receives 64 channels. A width of 10 with
3 heads is invalid because no integer per-head width preserves all ten
channels.


    ### Hints and common mistakes


Reject zero and negative dimensions before using modulo. Returning a
truncated integer division for a non-divisible width hides a later
reshape error and makes debugging harder.

In [ ]:
def validate_attention_dimensions(
    embedding_dim: int, num_heads: int
) -> int:
    # LLM_TODO_CH03_03_DIMENSIONS
    raise NotImplementedError(
        "validate divisibility and return per-head width"
    )

### Check your implementation

Run the next cell only after replacing the TODO above. A useful
check tests behavior, not just whether the function returns.

In [ ]:
assert validate_attention_dimensions(768, 12) == 64
try:
    validate_attention_dimensions(10, 3)
except ValueError:
    pass
else:
    raise AssertionError("an uneven split should fail")
print("Exercise 3.3 passed")

## Chapter 03 synthesis


- Why does `nn.Linear.weight` look transposed relative to $XW$ notation?
- Which tensor dimensions describe examples, positions, heads, and channels?
- Where would a wrong concatenation axis first become observable?


    After answering these questions, run the chapter's `solution.py`
    from a terminal to compare behavior. The reference file is compact
    by design; the reasoning in this notebook is the main lesson.